# Singling-Out Attack with Tree-Based Query Construction

This notebook demonstrates the enhanced singling-out privacy attack with decision tree-based query construction for testing synthetic data generators.

## Overview

This example showcases new features added to the singling-out attack:

- **Tree-based query construction** (`use_tree` parameter): Uses decision trees to generate smarter queries for continuous columns
- **LeakySynthesizer**: Utility for testing privacy with intentionally leaked data
- **Advanced parameter tuning**: Fine-grained control over attack behavior

These improvements significantly enhance the attack's effectiveness, especially for datasets with continuous numerical features.

In [ ]:
import os
import sys
import pandas as pd

sys.path.append("../..")

from leakpro.synthetic_data_attacks.anonymeter.evaluators.singling_out_evaluator import SinglingOutEvaluator
from leakpro.synthetic_data_attacks.leaky_synthesizer import LeakySynthesizer
from leakpro.synthetic_data_attacks.plots import plot_singling_out
from leakpro.synthetic_data_attacks.singling_out_utils import load_singling_out_results, singling_out_risk_evaluation

## Load Data

We'll use the Adults dataset from a public bucket for portability.

In [ ]:
bucket_url = "https://storage.googleapis.com/statice-public/anonymeter-datasets/"

# Load datasets
ori = pd.read_csv(os.path.join(bucket_url, "adults_train.csv"))
syn = pd.read_csv(os.path.join(bucket_url, "adults_syn_ctgan.csv"))
control = pd.read_csv(os.path.join(bucket_url, "adults_control.csv"))

print(f"Original data shape: {ori.shape}")
print(f"Synthetic data shape: {syn.shape}")
print(f"Control data shape: {control.shape}")

## Basic Example with Tree-Based Queries

The `use_tree=True` parameter enables decision tree-based query construction. This creates adaptive intervals for continuous columns that better capture the data distribution.

**Note**: This example uses a small subset for faster execution. Increase `n_attacks` for more reliable results.

In [ ]:
# Use smaller subset for demonstration
ori_small = ori.head(1000)
syn_small = syn.head(1000)

# Run singling-out evaluation with tree-based queries
results_basic = singling_out_risk_evaluation(
    dataset="adults_basic_tree",
    ori=ori_small,
    syn=syn_small,
    n_cols=2,
    n_attacks=100,
    verbose=True,
    save_results_json=False,
    use_tree=True,
    use_medians=False
)

print("\nBasic tree-based evaluation completed!")

In [ ]:
# Plot the results
plot_singling_out(sin_out_res=results_basic, high_res_flag=False)

## Testing Privacy with LeakySynthesizer

The `LeakySynthesizer` helps evaluate privacy by creating intentionally leaky synthetic data. It replaces a fraction of the synthetic data with real training records.

This simulates a privacy breach and helps measure how vulnerable a synthetic data generator is to singling-out attacks.

In [ ]:
# Create a leaky synthesizer
synthesizer = LeakySynthesizer(training_set=ori_small, control_set=syn_small)

# Generate leaky data with 50% of records leaked
leaky_data = synthesizer.generate_leaky_data(
    leak_fraction=0.5,
    random_state=42
)

print(f"Original training size: {synthesizer.training_size}")
print(f"Release set size: {synthesizer.release_size}")
print(f"Leaky data shape: {leaky_data.shape}")

### Evaluate Singling-Out Risk on Leaky Data

Now let's see how the singling-out attack performs on data with known privacy leakage.

In [ ]:
results_leaky = singling_out_risk_evaluation(
    dataset="adults_leaky_50pct",
    ori=ori_small,
    syn=leaky_data,  # Using the leaky synthetic data
    n_cols=2,
    n_attacks=100,
    verbose=True,
    save_results_json=False,
    use_tree=True
)

print("\nLeaky data evaluation (50%) completed!")
print("Note: Higher attack success rates indicate the privacy leakage is detectable.")

plot_singling_out(sin_out_res=results_leaky, high_res_flag=False)

# --- 80% Leakage Experiment ---

# Generate leaky data with 80% of records leaked
leaky_data_80 = synthesizer.generate_leaky_data(
    leak_fraction=0.8,
    random_state=42
)

results_leaky_80 = singling_out_risk_evaluation(
    dataset="adults_leaky_80pct",
    ori=ori_small,
    syn=leaky_data_80,
    n_cols=2,
    n_attacks=100,
    verbose=True,
    save_results_json=False,
    use_tree=True
)

print("\nLeaky data evaluation (80%) completed!")
plot_singling_out(sin_out_res=results_leaky_80, high_res_flag=False)

## Advanced Configuration: Custom Tree Parameters

You can fine-tune the decision tree behavior with `tree_params`. This is useful for:
- Controlling tree complexity (`max_depth`, `min_samples_leaf`)
- Ensuring reproducibility (`random_state`)
- Optimizing for specific data distributions

**Important**: Higher `max_attempts` and `max_per_combo` increase thoroughness but take longer to run.

In [ ]:
# Advanced configuration with custom tree parameters
results_advanced = singling_out_risk_evaluation(
    dataset="adults_advanced_tree",
    ori=ori_small,
    syn=syn_small,
    n_cols=3,
    n_attacks=50,  # Reduced for faster execution
    verbose=True,
    save_results_json=False,
    max_attempts=10_000,
    max_per_combo=10,
    sample_size_per_combo=5,
    max_rounds_no_progress=20,
    use_tree=True,
    tree_params={
        'min_samples_leaf': 1,    # Allow very specific intervals
        'max_depth': None,        # No depth limit
        'random_state': 42        # Reproducibility
    }
)

print("\nAdvanced evaluation completed!")

## Comparing Tree-Based vs Standard Approach

Let's compare the tree-based approach against the standard median-based approach.

In [ ]:
# Tree-based approach
results_with_tree = singling_out_risk_evaluation(
    dataset="adults_with_tree",
    ori=ori_small,
    syn=syn_small,
    n_cols=2,
    n_attacks=50,
    verbose=False,
    save_results_json=False,
    use_tree=True,
    use_medians=False
)

# Standard approach (no tree)
results_without_tree = singling_out_risk_evaluation(
    dataset="adults_without_tree",
    ori=ori_small,
    syn=syn_small,
    n_cols=2,
    n_attacks=50,
    verbose=False,
    save_results_json=False,
    use_tree=False,
    use_medians=True
)

print("Comparison completed!")
print("\nWith tree-based queries:")
print(f"  Attack results: {results_with_tree.res[0][:3]}")
print("\nWithout tree-based queries (standard):")
print(f"  Attack results: {results_without_tree.res[0][:3]}")

## Best Practices

### When to Use Tree-Based Approach (`use_tree=True`)

✅ **Recommended for:**
- Datasets with many continuous numerical columns
- When you need more diverse and adaptive queries
- When standard median-based approach gives poor results

❌ **May not help with:**
- Purely categorical data
- Very small datasets (< 100 rows)
- When computational resources are limited

### Parameter Tuning Tips

1. **`n_attacks`**: Start with 100-1000 for testing, use 10,000+ for production
2. **`max_attempts`**: Increase if not getting enough successful queries
3. **`tree_params['max_depth']`**: Use `None` for full trees, or limit to 5-10 for faster execution
4. **`max_per_combo`**: Increase for more thorough exploration of column combinations
5. **`use_medians`**: Set to `False` when using tree-based approach for best results

### Performance Considerations

- Tree fitting adds upfront cost (~seconds per continuous column)
- Larger `n_attacks` scales linearly with time
- Use smaller subsets for initial testing
- Consider parallelization for multiple evaluations

## Full Example with Larger Dataset

Here's a complete example with more attacks for reliable results. 

**⚠️ Warning**: This may take several minutes to run depending on your hardware.

In [ ]:
# Uncomment to run full evaluation (takes longer)
# results_full = singling_out_risk_evaluation(
#     dataset="adults_full_tree_example",
#     ori=ori.head(5000),  # Use more data
#     syn=syn.head(5000),
#     n_attacks=1000,      # More attacks
#     verbose=True,
#     save_results_json=True,
#     max_attempts=50_000,
#     max_per_combo=20,
#     sample_size_per_combo=10,
#     use_tree=True,
#     tree_params={
#         'min_samples_leaf': 1,
#         'max_depth': None,
#         'random_state': 42
#     }
# )
# 
# plot_singling_out(sin_out_res=results_full, high_res_flag=True)

print("Full example ready to run (uncomment above code)")

## Summary

This notebook demonstrated:

1. ✅ **Tree-based query construction** for more effective singling-out attacks
2. ✅ **LeakySynthesizer** for testing privacy with intentional data leakage  
3. ✅ **Advanced parameter tuning** for fine-grained control
4. ✅ **Comparison** between tree-based and standard approaches
5. ✅ **Best practices** for real-world usage

For more information, see:
- `leakpro/synthetic_data_attacks/leaky_synthesizer.py` - LeakySynthesizer implementation
- `leakpro/synthetic_data_attacks/singling_out_utils.py` - Main evaluation function
- `leakpro/synthetic_data_attacks/anonymeter/evaluators/singling_out_evaluator.py` - Core attack logic